# Patch-level Anomaly Detection + Defect-Type Classification (실험)

개념(discriminative FSAD / AnomalyDINO 계열)을 **UI에 넣기 전에** 검증하는 실험.

- **정상 patch 메모리**를 기준으로 각 patch 의 이상도(anomaly)를 계산.
- 불량 이미지에서 **정상 대비 이질적인 patch만** 골라 **불량종류별 bank** 구성.
- query 이미지: patch별 **(1) anomaly 여부** + **(2) 어떤 불량종류인지** 를 시각화.

> 서버(GPU)에서 실행. 백본을 직접 로드(서비스 불필요). feature_kind 는 patch 격자를 쓴다.

## 0. 경로 / 임포트

In [ ]:
import os, sys
from pathlib import Path
_HERE = Path.cwd(); _REPO = _HERE
while not (_REPO / 'dino_v3').exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
for p in (str(_REPO), str(_REPO / 'dino_v3')):
    if p not in sys.path:
        sys.path.insert(0, p)
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import dinov3.distributed as distributed
from dinov3.eval.setup import setup_and_build_model
from inspection.em_aug import build_em_eval_transform
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. 설정
`DATA_ROOT` = 클래스별 폴더(**normal 포함**). 예: `normal/ scratch/ stain/ ...`

In [ ]:
CONFIG      = 'dino_v3/dinov3/configs/train/weaksup/stage2_ssl_weaksup.yaml'  # FILL_IN
CKPT        = '/path/to/teacher_checkpoint.pth'   # FILL_IN
DATA_ROOT   = '/path/to/class_folders'            # FILL_IN (normal/ + 불량종류/)
NORMAL_CLASS= 'normal'
IMAGE_SIZE  = 448
N_BLOCKS    = 1        # 사용할 마지막 블록 수 (중간 레이어 실험은 아래 주석 참고)
MEM_CAP     = 20000    # 정상 patch 메모리 최대 크기(랜덤 subsample)
ANOM_PCTL   = 99.0     # 정상 calib patch 이상도의 몇 퍼센타일을 임계로
MIN_ANOM_FRAC = 0.003  # 이미지 판정: 이상 patch 비율이 이보다 크면 불량
MAX_IMG_PER_CLASS = 40 # 클래스당 최대 사용 이미지
SEED        = 0
CONFIG = str((_REPO / CONFIG)) if not os.path.isabs(CONFIG) else CONFIG
rng = np.random.default_rng(SEED)

## 2. 백본 로드 + patch feature 추출기

In [ ]:
if not distributed.is_enabled():
    distributed.enable(overwrite=True)
model, ctx = setup_and_build_model(config_file=CONFIG, pretrained_weights=CKPT, output_dir='./.em_cache')
model.cuda().eval()
autocast_dtype = ctx['autocast_dtype']
eval_tf = build_em_eval_transform(IMAGE_SIZE)
print('embed_dim', model.embed_dim)

@torch.inference_mode()
def patch_features(pil):
    """이미지 → (feats[h*w, D] L2정규화, h, w). 마지막 N_BLOCKS 블록 patch."""
    x = eval_tf(pil.convert('RGB')).unsqueeze(0).cuda()
    with torch.autocast('cuda', dtype=autocast_dtype):
        outs = model.get_intermediate_layers(x, n=N_BLOCKS, reshape=True, return_class_token=True, norm=True)
    grid = outs[-1][0][0].float().cpu().numpy()      # [C, h, w]
    C, h, w = grid.shape
    f = grid.reshape(C, h * w).T                     # [h*w, C]
    f = f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)
    return f.astype(np.float32), h, w

# 중간 레이어 실험(AnomalyDINO): model.get_intermediate_layers(x, n=[3,6,9], ...) 로 여러 층을
#   뽑아 concat/avg 하면 됨. 여기선 단순화를 위해 마지막 블록 사용.
IMG_EXTS = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def list_imgs(folder, cap=None):
    ps = [p for p in sorted(Path(folder).rglob('*')) if p.suffix.lower() in IMG_EXTS]
    if cap and len(ps) > cap:
        ps = [ps[i] for i in rng.choice(len(ps), cap, replace=False)]
    return ps

## 3. 정상 patch 메모리 + 이상 임계
정상 이미지를 memory/calib 로 나눠, **calib 이상도 분포**에서 임계를 잡는다(정상 대비 얼마나 벗어나야 이상인지).

In [ ]:
root = Path(DATA_ROOT)
class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
assert any(d.name == NORMAL_CLASS for d in class_dirs), f'{NORMAL_CLASS} 폴더 필요'
defect_names = [d.name for d in class_dirs if d.name != NORMAL_CLASS]
print('defect classes:', defect_names)

normal_paths = list_imgs(root / NORMAL_CLASS, MAX_IMG_PER_CLASS)
n_mem = max(1, int(len(normal_paths) * 0.7))
mem_paths, calib_paths = normal_paths[:n_mem], normal_paths[n_mem:]

def collect_patches(paths):
    feats = []
    for p in paths:
        f, h, w = patch_features(Image.open(p))
        feats.append(f)
    return np.vstack(feats) if feats else np.zeros((0, model.embed_dim), np.float32)

M_normal = collect_patches(mem_paths)
if len(M_normal) > MEM_CAP:
    M_normal = M_normal[rng.choice(len(M_normal), MEM_CAP, replace=False)]
print('정상 메모리 patch:', M_normal.shape)

def anomaly_scores(q):
    """q[Nq,D] → 1 - max cosine to M_normal (0=정상같음, 클수록 이상)."""
    # 청크로 계산(메모리 안전)
    out = np.empty(len(q), np.float32)
    for i in range(0, len(q), 4096):
        out[i:i+4096] = 1.0 - (q[i:i+4096] @ M_normal.T).max(axis=1)
    return out

calib = collect_patches(calib_paths) if calib_paths else M_normal
thr = float(np.percentile(anomaly_scores(calib), ANOM_PCTL))
print(f'이상 임계(anomaly thr) = {thr:.4f}  (정상 calib {ANOM_PCTL}퍼센타일)')

## 4. 불량종류별 patch bank (정상 대조 필터)
각 불량 폴더에서 patch 를 뽑되, **이상도 > 임계인 patch만** bank 에 담는다(배경 오염 방지).

In [ ]:
banks = []   # (name, arr[Nc, D])
for name in defect_names:
    paths = list_imgs(root / name, MAX_IMG_PER_CLASS)
    feats = collect_patches(paths)
    a = anomaly_scores(feats)
    kept = feats[a > thr]
    banks.append((name, kept))
    print(f'  {name}: patch {len(feats)} → 이상 patch {len(kept)} ({100*len(kept)/max(1,len(feats)):.1f}%) bank 등록')
assert all(len(b[1]) > 0 for b in banks), '임계가 너무 높아 bank 가 빈 클래스 있음 → ANOM_PCTL 낮추기'

## 5. 추론 — patch별 anomaly + 불량종류

In [ ]:
def classify_patches(q):
    """q[Nq,D] → (type_idx[Nq], type_sim[Nq]) : 불량 bank 중 최근접."""
    best = np.full(len(q), -1, int); bestsim = np.full(len(q), -1.0, np.float32)
    for ci, (_, arr) in enumerate(banks):
        s = (q @ arr.T).max(axis=1)
        m = s > bestsim; best[m] = ci; bestsim[m] = s[m]
    return best, bestsim

def analyze(pil):
    f, h, w = patch_features(pil)
    anom = anomaly_scores(f)                    # [h*w]
    is_anom = anom > thr
    ttype, tsim = classify_patches(f)
    type_map = np.where(is_anom, ttype, -1)  # 이상 patch만 종류 부여
    # 이미지 판정
    frac = is_anom.mean()
    if frac < MIN_ANOM_FRAC:
        img_pred = NORMAL_CLASS
    else:
        mass = {nm: float(tsim[(is_anom) & (ttype == ci)].sum()) for ci, (nm, _) in enumerate(banks)}
        img_pred = max(mass, key=mass.get)
    return dict(h=h, w=w, anom=anom, is_anom=is_anom, type_map=type_map, frac=float(frac), pred=img_pred)

## 6. 시각화 — 어떤 patch 가 이상이고, 어떤 불량인지
입력 | anomaly heatmap | 불량종류 overlay(이상 patch만 클래스 색).

In [ ]:
import matplotlib.colors as mcolors
CLASS_COLORS = plt.cm.tab10(np.arange(len(defect_names)) % 10)[:, :3]

def upsample(a2d, size):
    return np.asarray(Image.fromarray(a2d).resize((size, size), Image.NEAREST))

def show(pil, r, title=''):
    h, w = r['h'], r['w']
    base = np.asarray(pil.resize((IMAGE_SIZE, IMAGE_SIZE)).convert('L'), float) / 255
    anom_img = np.asarray(Image.fromarray((r['anom'].reshape(h, w)).astype(np.float32)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR))
    # 종류 overlay (이상 patch 만 색)
    tm = r['type_map'].reshape(h, w)
    over = np.zeros((h, w, 4), np.float32)
    for ci in range(len(defect_names)):
        over[tm == ci, :3] = CLASS_COLORS[ci]; over[tm == ci, 3] = 0.8
    over_img = np.asarray(Image.fromarray((over*255).astype(np.uint8)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)).astype(float)/255
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
    ax[0].imshow(base, cmap='gray'); ax[0].set_title(f'input\npred={r["pred"]}'); ax[0].axis('off')
    im1 = ax[1].imshow(anom_img, cmap='inferno'); ax[1].set_title(f'anomaly (thr={thr:.3f})'); ax[1].axis('off')
    fig.colorbar(im1, ax=ax[1], fraction=0.046)
    ax[2].imshow(base, cmap='gray'); ax[2].imshow(over_img); ax[2].set_title('defect type (이상 patch)'); ax[2].axis('off')
    handles = [plt.Line2D([0],[0], marker='s', ls='', color=CLASS_COLORS[ci], label=defect_names[ci]) for ci in range(len(defect_names))]
    ax[2].legend(handles=handles, fontsize=7, loc='upper right')
    plt.suptitle(title); plt.tight_layout(); plt.show()

# 각 불량 클래스 + 정상에서 몇 장씩 확인
for name in defect_names + [NORMAL_CLASS]:
    for p in list_imgs(root / name)[:2]:
        pil = Image.open(p)
        show(pil, analyze(pil), title=f'{name} / {Path(p).name}')

## 7. (옵션) 정량 — image-level 종류 정확도
support/query 로 나눠, patch 집계 예측의 이미지 단위 정확도(불량 클래스만).

In [ ]:
# 주의: 위 banks 는 전체로 만들었으므로, 엄밀한 평가는 support/query 분리 후 bank 재구성 필요.
# 여기선 간단 확인용: 각 불량 클래스의 이미지가 자기 클래스로 예측되는 비율.
correct = total = 0
conf = {n: {m: 0 for m in defect_names} for n in defect_names}
for name in defect_names:
    for p in list_imgs(root / name):
        r = analyze(Image.open(p))
        if r['pred'] in defect_names:
            conf[name][r['pred']] += 1
        total += 1; correct += int(r['pred'] == name)
print(f'불량 이미지 종류 정확도(전체 bank 기준, 낙관적): {100*correct/max(1,total):.1f}%')
print('혼동:', conf)
print('※ 엄밀 평가는 §7 을 support/query 분리로 다시 짜야 함.')